## 0. Dependencias

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import warnings
import joblib
import os
from sklearn.mixture import GaussianMixture

warnings.filterwarnings('ignore')

SEED = 20260424
np.random.seed(SEED)

## 1. Descarga de modelo original

In [2]:
model = joblib.load("datos/v6/modelos/M3_gbm_completo.pkl")

In [3]:
model

{'pipeline': Pipeline(steps=[('pre',
                  ColumnTransformer(transformers=[('num', 'passthrough',
                                                   ['monto_log', 'secuencia_24h',
                                                    'terminal_risk_score']),
                                                  ('cat',
                                                   OneHotEncoder(handle_unknown='ignore'),
                                                   ['canal', 'dispositivo',
                                                    'hora_bin'])])),
                 ('clf',
                  GradientBoostingClassifier(learning_rate=0.08, max_depth=4,
                                             n_estimators=200,
                                             random_state=20260424,
                                             subsample=0.8))]),
 'features': ['monto_log',
  'secuencia_24h',
  'terminal_risk_score',
  'canal',
  'dispositivo',
  'hora_bin'],
 'threshold': 0.46}

## 2. Carga de datos etiquetados con cluster

- `es_fraude` (bool)
- `cluster_fraude` (0 para fraudes tipicos en T0 y 1 para nuevos fraudes en T1)

In [4]:
df = pd.read_parquet("datos/v6/operaciones.parquet")
df = df[df['etiquetado'] == True]

df_cluster = pd.read_parquet("datos/v6/clustering/clustering_operaciones.parquet")
df_cluster = df_cluster[df_cluster['cluster_fraude'].notna()]

df_merger = df.merge(df_cluster[["op_id", "cluster_fraude"]], on="op_id", how="inner")

In [5]:
df_merger['etiquetado'].value_counts()

etiquetado
True    5786
Name: count, dtype: int64

In [6]:
assert 'es_fraude' in df_merger.columns
assert 'cluster_fraude' in df_merger.columns

n_fraude = df_merger['es_fraude'].sum()
n_legitimas = (~df_merger['es_fraude']).sum()
n_tipo0 = (df_merger['cluster_fraude'] == 0).sum()
n_tipo1 = (df_merger['cluster_fraude'] == 1).sum()

print(f"Total registros : {len(df_merger):,}")
print(f"Fraudes totales : {n_fraude:,}  ({n_fraude/len(df_merger)*100:.2f}%)")
print(f"  cluster 0     : {n_tipo0:,}")
print(f"  cluster 1     : {n_tipo1:,}")
print(f"Legítimas       : {n_legitimas:,}")

Total registros : 5,786
Fraudes totales : 5,786  (100.00%)
  cluster 0     : 4,781
  cluster 1     : 1,005
Legítimas       : 0


In [7]:
pipeline = model['pipeline']

preprocesador = pipeline.named_steps['pre']
columnas_esperadas = preprocesador.feature_names_in_

X_crudo = df_merger[columnas_esperadas]
y = df_merger['cluster_fraude']

X_train, X_test, y_train, y_test = train_test_split(
    X_crudo,
    y,
    test_size=0.05,
    random_state=20260424,
    stratify=y
)

print(f"Tamaño Entrenamiento: {X_train.shape[0]} filas")
print(f"Tamaño Testeo:        {X_test.shape[0]} filas")
print(f"Proporción de fraude en Testeo: {y_test.mean():.4f}")


Tamaño Entrenamiento: 5496 filas
Tamaño Testeo:        290 filas
Proporción de fraude en Testeo: 0.1724


In [42]:
# =====================================================================
# 1. SEPARAR LOS DATOS POR TIPO DE FRAUDE
# =====================================================================
# Filtramos X_train usando la máscara del target real de fraude
X_train_t0 = X_train[y_train == 0]
X_train_t1 = X_train[y_train == 1]

# =====================================================================
# 2. DEFINIR LA GRILLA DE HIPERPARÁMETROS PARA GMM
# =====================================================================
param_grid_gmm = {
    'clf__n_components': [2, 3, 5, 7],
    'clf__covariance_type': ['full', 'tied', 'diag']
}

# =====================================================================
# 3. CONFIGURAR Y ENTRENAR EL MODELO C1 (FRAUDE BASE T0)
# =====================================================================
pipeline_gmm_t0 = Pipeline(steps=[
    ('pre', preprocesador),
    ('clf', GaussianMixture(random_state=20260424, max_iter=200))
])

grid_search_base = GridSearchCV(
    estimator=pipeline_gmm_t0,
    param_grid=param_grid_gmm,
    cv=5,
    n_jobs=-1,
    verbose=1
)

print("Entrenando GMM para Fraude Base (T0)...")
grid_search_base.fit(X_train_t0)

# =====================================================================
# 4. CONFIGURAR Y ENTRENAR EL MODELO C2 (FRAUDE SMISHING T1)
# =====================================================================
pipeline_gmm_t1 = Pipeline(steps=[
    ('pre', preprocesador),
    ('clf', GaussianMixture(random_state=20260424, max_iter=200))
])

grid_search_sm = GridSearchCV(
    estimator=pipeline_gmm_t1,
    param_grid=param_grid_gmm,
    cv=5,
    n_jobs=-1,
    verbose=1
)

print("Entrenando GMM para Fraude de Smishing (T1)...")
grid_search_sm.fit(X_train_t1)  # Ojo: Tampoco lleva 'y'

print("\n¡Entrenamiento completo de ambos modelos generativos!")

Entrenando GMM para Fraude Base (T0)...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Entrenando GMM para Fraude de Smishing (T1)...
Fitting 5 folds for each of 12 candidates, totalling 60 fits

¡Entrenamiento completo de ambos modelos generativos!


## 7. Inferencia sobre transacciones nuevas

### 7a. Posteriors y verosimilitudes

Para recuperar la **verosimilitud** P(x | tipo_k) a partir de la posterior
que devuelve `predict_proba`, corregimos por el prior de entrenamiento:

```
P(x | tipo_k)  ∝  P(tipo_k | x)  /  prior_pos_k
```

Esto elimina el sesgo de prevalencia absorbido durante el entrenamiento.

In [52]:
def score_transaccion_grid(x_row: pd.DataFrame,
                           grid_t0: dict,
                           grid_t1: dict,
                           w_tipo0: float = 0.5,
                           w_tipo1: float = 0.5) -> dict:
    """
    Calcula los scores generativos para una única transacción evaluando la
    verosimilitud absoluta y la sorpresa de Shannon mediante Gaussian Mixture Models.
    """
    assert abs(w_tipo0 + w_tipo1 - 1.0) < 1e-6, "w_tipo0 + w_tipo1 debe ser 1"

    pipeline_t0 = grid_t0['grid'].best_estimator_
    pipeline_t1 = grid_t1['grid'].best_estimator_

    cols_necesarias = pipeline_t1.named_steps['pre'].feature_names_in_
    x_input = x_row[cols_necesarias]

    log_like0 = pipeline_t0.score_samples(x_input)[0]
    log_like1 = pipeline_t1.score_samples(x_input)[0]

    prior0 = grid_t0['prior_pos']
    prior1 = grid_t1['prior_pos']

    like0 = np.exp(log_like0)
    like1 = np.exp(log_like1)

    LR = like1 / like0 if like0 > 0 else np.inf

    p_evento_modelo = (w_tipo0 * like0) + (w_tipo1 * like1)

    eps = 1e-10  # Previene indeterminaciones si p_evento_modelo colapsa a 0
    surprise_score = -np.log(p_evento_modelo + eps)

    alerta0 = log_like0 >= grid_t0['threshold']
    alerta1 = log_like1 >= grid_t1['threshold']

    alerta_sorpresa = surprise_score > 4.0

    alerta_any = alerta0 or alerta1 or alerta_sorpresa

    return {
        'log_like_tipo0': round(log_like0, 4),
        'log_like_tipo1': round(log_like1, 4),
        'like_tipo0': round(like0, 6),
        'like_tipo1': round(like1, 6),
        'LR(tipo1/tipo0)': round(LR, 4),
        'surprise_score_nats': round(surprise_score, 4),
        'alerta_tipo0': bool(alerta0),
        'alerta_tipo1': bool(alerta1),
        'alerta_sorpresa': bool(alerta_sorpresa),
        'alerta_any': bool(alerta_any)
    }

In [53]:
prior_base = (y_train == 0).mean()
prior_smishing = (y_train == 1).mean()

modelo_t0_dict = {
    'grid': grid_search_base,
    'threshold': -15.0,
    'prior_pos': prior_base
}

modelo_t1_dict = {
    'grid': grid_search_sm,
    'threshold': -15.0,
    'prior_pos': prior_smishing
}

fila_nueva = X_test.iloc[[0]]

res = score_transaccion_grid(fila_nueva, modelo_t0_dict, modelo_t1_dict, w_tipo0=0.5, w_tipo1=0.5)

print("Resultados del Enrutador Generativo GMM:")
print("-" * 40)
for k, v in res.items():
    print(f"{k}: {v}")

Resultados del Enrutador Generativo GMM:
----------------------------------------
log_like_tipo0: 24.8841
log_like_tipo1: 27.6869
like_tipo0: 64125814840.24886
like_tipo1: 1057424471893.8962
LR(tipo1/tipo0): 16.4898
surprise_score_nats: -27.0526
alerta_tipo0: True
alerta_tipo1: True
alerta_sorpresa: False
alerta_any: True


In [45]:
fila_nueva

,monto_log,secuencia_24h,terminal_risk_score,canal,dispositivo,hora_bin
5427,9.993862,3,0.250119,online,registrado_reciente,manana


In [46]:
y_test.iloc[[0]]

5427    1.0
Name: cluster_fraude, dtype: float64

### 7b. Scoring en batch sobre un DataFrame completo

In [54]:
def score_batch_grid(df_nuevas: pd.DataFrame,
                     grid_t0: dict,
                     grid_t1: dict,
                     w_tipo0: float = 0.5,
                     w_tipo1: float = 0.5) -> pd.DataFrame:
    """
    Calcula scores masivos para un lote de transacciones evaluando la sorpresa
    de Shannon unificada mediante modelos generativos Gaussian Mixture (GMM).

    Parámetros
    ----------
    df_nuevas    : DataFrame con los datos crudos/originales de las nuevas operaciones
    grid_t0      : dict del modelo Base C1 {'grid': grid_search_base, 'threshold': -15.0, 'prior_pos': p0}
    grid_t1      : dict del modelo Smishing C2 {'grid': grid_search_sm, 'threshold': -15.0, 'prior_pos': p1}
    w_tipo0      : peso/prevalencia del tipo 0 en el mes actual
    w_tipo1      : peso/prevalencia del tipo 1 en el mes actual
    """
    assert abs(w_tipo0 + w_tipo1 - 1.0) < 1e-6, "Los pesos w deben sumar 1"

    pipeline_t0 = grid_t0['grid'].best_estimator_
    pipeline_t1 = grid_t1['grid'].best_estimator_

    cols_necesarias = pipeline_t1.named_steps['pre'].feature_names_in_
    df_input = df_nuevas[cols_necesarias]

    log_like0 = pipeline_t0.score_samples(df_input)
    log_like1 = pipeline_t1.score_samples(df_input)

    like0 = np.exp(log_like0)
    like1 = np.exp(log_like1)

    LR = np.divide(
        like1,
        like0,
        out=np.full_like(like1, np.inf),
        where=like0 > 0
    )

    p_evento_modelo = (w_tipo0 * like0) + (w_tipo1 * like1)

    eps = 1e-10
    surprise_score = -np.log(p_evento_modelo + eps)

    alerta0 = log_like0 >= grid_t0['threshold']
    alerta1 = log_like1 >= grid_t1['threshold']

    alerta_sorpresa = surprise_score > 4.0

    alerta_any = alerta0 | alerta1 | alerta_sorpresa

    return pd.DataFrame({
        'log_like_tipo0': np.round(log_like0, 4),
        'log_like_tipo1': np.round(log_like1, 4),
        'like_tipo0': np.round(like0, 6),
        'like_tipo1': np.round(like1, 6),
        'LR(tipo1/tipo0)': np.round(LR, 4),
        'surprise_score_nats': np.round(surprise_score, 4),
        'alerta_tipo0': alerta0,
        'alerta_tipo1': alerta1,
        'alerta_sorpresa': alerta_sorpresa,
        'alerta_any': alerta_any
    }, index=df_nuevas.index)

In [55]:
df_scores_test = score_batch_grid(
    df_nuevas=X_test,
    grid_t0=modelo_t0_dict,
    grid_t1=modelo_t1_dict,
    w_tipo0=0.5,
    w_tipo1=0.5
)

print(f"Alertas totales (Cualquier origen): {df_scores_test['alerta_any'].sum()} ({df_scores_test['alerta_any'].mean()*100:.2f}%)")
print(f"Alertas por SORPRESA (Régimen Desconocido): {df_scores_test['alerta_sorpresa'].sum()} ({df_scores_test['alerta_sorpresa'].mean()*100:.2f}%)")

df_scores_test

Alertas totales (Cualquier origen): 290 (100.00%)
Alertas por SORPRESA (Régimen Desconocido): 0 (0.00%)


,log_like_tipo0,log_like_tipo1,like_tipo0,like_tipo1,LR(tipo1/tipo0),surprise_score_nats,alerta_tipo0,alerta_tipo1,alerta_sorpresa,alerta_any
5427,24.8841,27.6869,6.412581e+10,1.057424e+12,16.4898,-27.0526,True,True,False,True
2612,27.4897,31.2114,8.681845e+11,3.588726e+13,41.3360,-30.5422,True,True,False,True
1134,29.0975,20.8504,4.333843e+12,1.135595e+09,0.0003,-28.4046,True,True,False,True
4640,33.7628,35.3787,4.602415e+14,2.316125e+15,5.0324,-34.8668,True,True,False,True
1137,31.8252,25.6767,6.630132e+13,1.416593e+11,0.0021,-31.1342,True,True,False,True
...,...,...,...,...,...,...,...,...,...,...
5554,28.3118,35.2292,1.975488e+12,1.994539e+15,1009.6435,-34.5370,True,True,False,True
2508,29.8783,37.2781,9.462298e+12,1.547578e+16,1635.5198,-36.5855,True,True,False,True
3549,30.1852,31.2346,1.286109e+13,3.672855e+13,2.8558,-30.8417,True,True,False,True
1358,25.4838,25.5583,1.168033e+11,1.258408e+11,1.0774,-25.5217,True,True,False,True


In [56]:
df_scores_test[df_scores_test['surprise_score_nats'] >= 4]

,log_like_tipo0,log_like_tipo1,like_tipo0,like_tipo1,LR(tipo1/tipo0),surprise_score_nats,alerta_tipo0,alerta_tipo1,alerta_sorpresa,alerta_any


In [57]:
df_scores_M19 = score_batch_grid(
    df_nuevas=df[(df['etiquetado'] == True) & (df['mes'] == 19)][columnas_esperadas],
    grid_t0=modelo_t0_dict,
    grid_t1=modelo_t1_dict,
    w_tipo0=0.5,
    w_tipo1=0.5
)

df_scores_M19

,log_like_tipo0,log_like_tipo1,like_tipo0,like_tipo1,LR(tipo1/tipo0),surprise_score_nats,alerta_tipo0,alerta_tipo1,alerta_sorpresa,alerta_any
900239,16.1893,24.1696,1.073829e+07,3.138494e+10,2.922713e+03,-23.4768,True,True,False,True
900240,22.6811,26.8896,7.083914e+09,4.764521e+11,6.725830e+01,-26.2112,True,True,False,True
900241,13.5353,26.9008,7.556038e+05,4.817897e+11,6.376221e+05,-26.2076,True,True,False,True
900244,24.2927,23.7372,3.549500e+10,2.036642e+10,5.738000e-01,-24.0530,True,True,False,True
900248,17.6494,21.8583,4.624069e+07,3.111198e+09,6.728270e+01,-21.1799,True,True,False,True
...,...,...,...,...,...,...,...,...,...,...
947582,24.3037,26.7979,3.589012e+10,4.346845e+11,1.211150e+01,-26.1841,True,True,False,True
947584,20.3566,38.6089,6.930575e+08,5.856679e+16,8.450495e+07,-37.9158,True,True,False,True
947585,15.9606,21.0084,8.542973e+06,1.329942e+09,1.556767e+02,-20.3217,True,True,False,True
947586,25.7274,26.8105,1.490223e+11,4.402173e+11,2.954000e+00,-26.4090,True,True,False,True


In [40]:
df[(df['etiquetado'] == True) & (df['mes'] == 19)]

,op_id,mes,canal,hora_bin,dispositivo,monto_log,terminal_risk_score,secuencia_24h,etiquetado,es_fraude
900239,900240,19,presencial,tarde,conocido,7.944736,0.130966,2,True,False
900240,900241,19,presencial,tarde,conocido,9.115504,0.229485,4,True,False
900241,900242,19,transferencia,noche,conocido,9.490015,0.071279,2,True,False
900244,900245,19,online,tarde,conocido,7.659744,0.506703,2,True,False
900248,900249,19,online,manana,conocido,7.430267,0.184106,1,True,False
...,...,...,...,...,...,...,...,...,...,...
947582,947583,19,online,manana,conocido,8.750480,0.278023,5,True,False
947584,947585,19,atm,tarde,conocido,7.919713,0.282081,4,True,False
947585,947586,19,presencial,manana,conocido,7.232501,0.116362,5,True,False
947586,947587,19,presencial,tarde,conocido,8.578720,0.347487,3,True,False


In [58]:
df_scores_M19[df_scores_M19['surprise_score_nats'] >= 4]

,log_like_tipo0,log_like_tipo1,like_tipo0,like_tipo1,LR(tipo1/tipo0),surprise_score_nats,alerta_tipo0,alerta_tipo1,alerta_sorpresa,alerta_any


In [20]:
os.makedirs('datos/v6/modelos/clasificacion_fraude', exist_ok=True)

nuevos_modelos_guardar = {
    'C1_gbm_base': {
        'grid': grid_search_base,
        'threshold': -15.0,
        'prior_pos': (y_train == 0).mean()
    },
    'C2_gbm_smishing': {
        'grid': grid_search_sm,
        'threshold': -15.0,
        'prior_pos': (y_train == 1).mean()
    }
}

for key, obj in nuevos_modelos_guardar.items():
    path = f'datos/v6/modelos/clasificacion_fraude/{key}.pkl'

    joblib.dump(obj, path)
    print(f"Guardado exitosamente: {path}")

print("\nSerialización completa con Joblib.")

Guardado exitosamente: datos/v6/modelos/clasificacion_fraude/C1_gbm_base.pkl
Guardado exitosamente: datos/v6/modelos/clasificacion_fraude/C2_gbm_smishing.pkl

Serialización completa con Joblib.
